# Day 8 — Flat Staking and Simulation Inspection

Purpose: inspect simulated flat-stake bets created by `scripts/build_bet_simulation.py`.

No advanced backtest metrics here. Just sanity checks, because apparently restraint is a feature.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
BETS_PATH = ROOT / 'data' / 'processed' / 'simulated_bets.parquet'
REPORT_PATH = ROOT / 'outputs' / 'evaluation' / 'bet_simulation_report.csv'

In [ ]:
bets = pd.read_parquet(BETS_PATH)
report = pd.read_csv(REPORT_PATH)

bets.shape, report.shape

In [ ]:
bets.head()

In [ ]:
bets.columns.tolist()

In [ ]:
report

In [ ]:
assert bets['selected'].all()
assert bets['stake'].eq(1.0).all()
print('All simulated bets are selected rows with flat 1.0 unit stake.')

In [ ]:
expected_is_win = bets['outcome'].eq(bets['full_time_result'])
assert bets['is_win'].equals(expected_is_win)
print('Win flags match outcome == full_time_result.')

In [ ]:
winning_bets = bets.loc[bets['is_win']]
losing_bets = bets.loc[~bets['is_win']]

assert (winning_bets['profit'].round(10) == (winning_bets['stake'] * (winning_bets['odds'] - 1)).round(10)).all()
assert (losing_bets['profit'].round(10) == (-losing_bets['stake']).round(10)).all()
assert winning_bets['return'].round(10).equals((winning_bets['stake'] * winning_bets['odds']).round(10))
assert losing_bets['return'].eq(0).all()

print('Profit and return formulas are valid.')

In [ ]:
bets['is_win'].value_counts(dropna=False)

In [ ]:
bets.groupby('outcome').agg(
    n_bets=('match_id', 'size'),
    n_wins=('is_win', 'sum'),
    total_profit=('profit', 'sum'),
).reindex(['H', 'D', 'A'])

In [ ]:
bets.assign(cumulative_profit=bets['profit'].cumsum()).plot(x='match_date', y='cumulative_profit')
plt.title('Cumulative Simulated Profit — Flat Stake')
plt.xlabel('Match date')
plt.ylabel('Profit units')
plt.show()